In [105]:
import pandas as pd
url = "https://raw.githubusercontent.com/mricardo89/data-mining/main/Unit-II/datasets/googleplaystore.csv"
df = pd.read_csv(url)

Pregunta A

Empezamos conociendo cuantos registros hay

In [55]:
antes = df.shape[0]
print(f"Hay {antes} registros")


Hay 10841 registros


In [56]:
df.duplicated(subset=['App']).sum()

np.int64(1181)

Hay 1181 duplicados, por lo que aplicaremos una funcion para eliminarlos y quedarnos unciamente con la primer aparicion de c/u.

In [57]:
df.drop_duplicates(subset=['App'], keep='first', inplace=True) 

In [58]:
despues = df.shape[0]
registros_eliminados = antes - despues
print(f"Se eliminaron {registros_eliminados} registros")

Se eliminaron 1181 registros


--> Se purgaron un total de 1181 registros (la totalidad de registros duplicados)

In [59]:
df['Installs-clean'] = df['Installs'].astype(str).str.replace('+', '', regex=False)
df['Installs-clean'] = df['Installs-clean'].str.replace(',', '', regex=False)

df['Installs-clean'].unique()

array(['10000', '500000', '5000000', '50000000', '100000', '50000',
       '1000000', '10000000', '5000', '100000000', '1000000000', '1000',
       '500000000', '50', '100', '500', '10', '1', '5', '0', 'Free'],
      dtype=object)

Como podemos observar, hay un registro "Free" en la columna Installs, hay que eliminarlo con un replace

In [60]:
df['Installs-clean'] = df['Installs-clean'].str.replace('Free', '0')

Quitamos espacios en blanco

In [61]:
df['Installs-clean'] = df['Installs-clean'].str.strip()

Como hay valores '', no se puede castear, por lo que los convertiremos a NaN

In [62]:
df['Installs-clean'] = pd.to_numeric(df['Installs-clean'], errors='coerce')


In [63]:
df['Installs-clean'].astype(float)

0           10000.0
1          500000.0
2         5000000.0
3        50000000.0
4          100000.0
            ...    
10836        5000.0
10837         100.0
10838        1000.0
10839        1000.0
10840    10000000.0
Name: Installs-clean, Length: 9660, dtype: float64

Calculando el promedio de descargas

In [93]:
df['Installs-clean'].mean()

np.float64(7776701.607349897)

C:
--> El promedio es de 7777506.732270421 descargas

Quitamos signos de '$' para leer precios 

In [113]:
df['Price'].head()

0    0
1    0
2    0
3    0
4    0
Name: Price, dtype: object

In [114]:
df['Price-clean'] = df['Price'].str.replace('$', '', regex=False)

In [115]:
df['Price-clean'] = df['Price-clean'].str.replace(',', '', regex=False)

Hay un registro que dice 'everyone'. Lo reemplazamos

In [116]:
df['Price-clean'] = df['Price-clean'].str.replace('Everyone', '', regex=False)

Quitamos espacios en blanco y convertimos a NaN los reemplazos a '' 

In [117]:
df['Price-clean'] = df['Price-clean'].str.strip()
df['Price-clean'] = pd.to_numeric(df['Price-clean'], errors='coerce')

In [118]:
df['Price-clean'].isna().sum()

np.int64(1)

In [119]:
df['Price-clean'].astype(float)

0        0.0
1        0.0
2        0.0
3        0.0
4        0.0
        ... 
10836    0.0
10837    0.0
10838    0.0
10839    0.0
10840    0.0
Name: Price-clean, Length: 10841, dtype: float64

En este momento no entendía porque solo habían 0s, pero al ejecutar value counts:

In [121]:
df['Price-clean'].value_counts()

Price-clean
0.00     10040
0.99       148
2.99       129
1.99        73
4.99        72
         ...  
19.90        1
1.75         1
14.00        1
4.85         1
1.04         1
Name: count, Length: 92, dtype: int64

Vemos que la gran mayoria son gratis

In [122]:
df['Price-clean'].max()

400.0

Efectivamente, la app mas cara es de 400.

Buscamos cuales son estas apps (arriba de los 200) y filtramos para unicamente ver la app y el rpecio

In [125]:

df[df['Price-clean'] > 200][['App', 'Price-clean']]

,App,Price-clean
4197,most expensive app (H),399.99
4362,💎 I'm rich,399.99
4367,I'm Rich - Trump Edition,400.00
5351,I am rich,399.99
5354,I am Rich Plus,399.99
5355,I am rich VIP,299.99
5356,I Am Rich Premium,399.99
5357,I am extremely Rich,379.99
5358,I am Rich!,399.99
5359,I am rich(premium),399.99


Sobreescribiendo el df para eliminar apps que meten sesgo.

In [126]:
df = df[df['Price-clean']<50]

In [127]:
df.to_csv('playstore_limpio.csv', index=False)